# Construction fichier consolidé

Le but de ce notebook est :
- De charger les données météo et phénologie préalablement préparés
- De préparer des trajectoires de données sur 365 jours
- De compléter le jeu de données en faisant du feature engineering
- De sauvegarder ce fichier consolidé qui sera le jeu de données source pour l'entraînement des modèles

In [1]:
#Librairies
import pandas as pd
import numpy as np
from datetime import datetime
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

In [ ]:
#Configuration
srcFolder = "../data/processed/consolidated/"
dstFolder = "../data/processed/consolidated/"

## Chargement des données météorologie et phénologie

In [3]:
pd_phenologie = pd.read_csv(srcFolder+"phenologie.csv")

In [4]:
pd_meteo = pd.read_parquet(srcFolder+"meteo.parquet")

## Préparation des données consolidées

In [5]:
#On ne garde que les colonnes qui nous intéressent sur le jeu de données phénologie
pd_all = pd_phenologie[["date", "annee", "jour_de_l_annee", "latitude_du_site", "longitude_du_site", "altitude_du_site_ign"]].copy()

In [6]:
pd_all

,date,annee,jour_de_l_annee,latitude_du_site,longitude_du_site,altitude_du_site_ign
0,2015-06-01,2015,152,47.96686,7.25566,432.0
1,2017-06-12,2017,163,44.65218,0.03119,67.0
2,2020-05-11,2020,132,44.65218,0.03119,67.0
3,2020-05-11,2020,132,44.65218,0.03119,67.0
4,2020-05-11,2020,132,44.65218,0.03119,67.0
...,...,...,...,...,...,...
48829,2022-05-23,2022,143,45.21006,-0.17011,70.0
48830,2022-05-17,2022,137,45.77756,-0.42825,63.0
48831,2022-05-24,2022,144,45.77756,-0.42825,63.0
48832,2022-05-24,2022,144,45.77756,-0.42825,63.0


In [7]:
#On ajoute la colonne date int (utilisée dans les calculs ultérieurs)
pd_all['date_int'] = pd_all['date'].astype(str).str.replace('-', '').astype(int)

#Dédoublonner
pd_all = pd_all.drop_duplicates(subset=['latitude_du_site', 'longitude_du_site', 'altitude_du_site_ign', 'annee', 'jour_de_l_annee'])

In [8]:
"""
    Calcule la distance en kilomètres entre deux points géographiques
"""
def haversine_distance(lat1, lon1, lat2, lon2):
   
    # Rayon de la Terre en km
    R = 6371.0
    
    # Conversion en radians
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    # Formule de Haversine
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    distance = R * c
    
    return distance

"""
    Calcule le numéro du jour dans l'année
    date_str: chaîne au format "AAAAMMJJ"
    retourne: numéro du jour (1-366)
"""
def jour_de_annee(date_str):
    date = datetime.strptime(str(int(date_str)), "%Y%m%d")
    return date.timetuple().tm_yday

In [9]:
#Fonction de génération des données consolidées

def create_lines(row):
    annee = int(row["annee"])
    dateStart = annee*10000+101
    dateEnd = int(row["date_int"])
    dateFinAnnee = annee*10000+1231
    dateChillingStart = (annee-1)*10000+1101  # 1er novembre de l'année précédente

    latA = row["latitude_du_site"]
    longA = row["longitude_du_site"]
    altA = row["altitude_du_site_ign"]

    test = pd_meteo[(pd_meteo["AAAAMMJJ"] <= dateFinAnnee) & (pd_meteo["AAAAMMJJ"] >= dateStart)].sort_values("AAAAMMJJ").copy()

    #On calcule la distance
    test['distance'] = haversine_distance(latA, longA, test['LAT'], test['LON'])

    #On recherche le point le plus proche
    points_plus_proches = pd_meteo.loc[test.groupby('AAAAMMJJ')['distance'].idxmin()]

    #On calcule la T moye
    points_plus_proches['T_moy'] = (points_plus_proches['TX'] + points_plus_proches['TN']) / 2

    T_base = 10
    points_plus_proches['GDD10'] = points_plus_proches['T_moy'].apply(lambda x: max(0, x - T_base))
    points_plus_proches['GDD10_cumul'] = points_plus_proches['GDD10'].cumsum()

    points_plus_proches['jour_n'] = range(1, len(points_plus_proches) + 1)
    points_plus_proches

    points_plus_proches = points_plus_proches.reset_index(drop=True)

    # Calcul du chilling hivernal (depuis le 1er novembre de l'année précédente)
    # Modèle simple : 1 unité si 0°C < T_moy <= 7.2°C
    test_chilling = pd_meteo[(pd_meteo["AAAAMMJJ"] <= dateFinAnnee) & (pd_meteo["AAAAMMJJ"] >= dateChillingStart)].sort_values("AAAAMMJJ").copy()
    test_chilling['distance'] = haversine_distance(latA, longA, test_chilling['LAT'], test_chilling['LON'])
    points_chilling = pd_meteo.loc[test_chilling.groupby('AAAAMMJJ')['distance'].idxmin()].copy()
    points_chilling['T_moy_c'] = (points_chilling['TX'] + points_chilling['TN']) / 2
    points_chilling['chilling'] = points_chilling['T_moy_c'].apply(lambda x: 1 if 0 < x <= 7.2 else 0)
    points_chilling['chilling_cumul'] = points_chilling['chilling'].cumsum()
    chilling_dict = dict(zip(points_chilling['AAAAMMJJ'], points_chilling['chilling_cumul']))

    col_latitude = []
    col_longitude = []
    col_altitude = []
    col_n_avant_floraison = []
    col_jour_n = []
    col_temps_thermique10 = []
    col_chilling_hivernal = []
    col_date = []

    jour_n_theorique = jour_de_annee(str(dateEnd))

    for index, row2 in points_plus_proches.iterrows():
        jour_actuel = jour_de_annee(row2['AAAAMMJJ'])
        col_latitude.append(latA)
        col_longitude.append(longA)
        col_altitude.append(altA)
        col_n_avant_floraison.append(jour_n_theorique - jour_actuel)  # Devient négatif après la floraison
        col_jour_n.append(index+1)
        col_temps_thermique10.append(row2['GDD10_cumul'])
        col_chilling_hivernal.append(chilling_dict.get(row2['AAAAMMJJ'], 0))
        col_date.append(row2["AAAAMMJJ"])

    dfAdd = pd.DataFrame({
        'latitude' : col_latitude,
        'longitude' : col_longitude,
        'altitude' : col_altitude,
        'n_avant_floraison' : col_n_avant_floraison,
        'jour_n' : col_jour_n,
        'temps_thermique10' : col_temps_thermique10,
        'chilling_hivernal' : col_chilling_hivernal,
        'date' : col_date,
        'annee' : annee
    })

    return dfAdd

## Enregistrement du fichier

In [10]:
output_path = dstFolder + "consolidated.parquet"

writer = None
try:
    for index, row in tqdm(pd_all.iterrows(), total=len(pd_all), desc="Construction du fichier consolidé"):
        df_chunk = create_lines(row)
        table = pa.Table.from_pandas(df_chunk)
        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema)
        writer.write_table(table)
finally:
    if writer is not None:
        writer.close()

print(f"Fichier enregistré : {output_path}")

Construction du fichier consolidé:   0%|          | 0/4754 [00:00<?, ?it/s]

Fichier enregistré : ../data/processed/consolidated2/consolidated.parquet
